In [30]:
from pathlib import Path
ROOT = Path().resolve().parent


In [31]:
import os

path = ROOT / "Main_Parquet_files"

files = []

for file in os.listdir(path):
    files.append(file)

In [32]:
import os
import pyarrow.parquet as pq

parquet_dir = ROOT / "Main_Parquet_files"


files = sorted([f for f in os.listdir(parquet_dir) if f.endswith(".parquet")])

for f in files:
    path = os.path.join(parquet_dir, f)
    pf = pq.ParquetFile(path)
    cols = pf.schema.names
    print(f"\n=== {f} ===")
    print(cols)


=== ComputedElements.parquet ===
["'PcrKey'", "'USCensusRegion'", "'USCensusDivision'", "'NasemsoRegion'", "'Urbanicity'", "'ageinyear'", "'EMSDispatchCenterTimeSec'", "'EMSChuteTimeMin'", "'EMSSystemResponseTimeMin'", "'EMSSceneResponseTimeMin'", "'EMSSceneTimeMin'", "'EMSSceneToPatientTimeMin'", "'EMSTransportTimeMin'", "'EMSTotalCallTimeMin'"]

=== EINJURY_01REF.parquet ===
["'eInjury_01'", "'DiagnosisCodeDescr'"]

=== EPROCEDURES_03REF.parquet ===
["'eProcedures_03'", "'ProcedureCodeDescr'"]

=== ESITUATION_09REF.parquet ===
["'eSituation_09'", "'DiagnosisCodeDescr'"]

=== ESITUATION_10REF.parquet ===
["'eSituation_10'", "'DiagnosisCodeDescr'"]

=== ESITUATION_11REF.parquet ===
["'eSituation_11'", "'DiagnosisCodeDescr'"]

=== ESITUATION_12REF.parquet ===
["'eSituation_12'", "'DiagnosisCodeDescr'"]

=== FACTPCRADDITIONALRESPONSEMODE.parquet ===
["'PcrKey'", "'eResponse_24'"]

=== FACTPCRADDITIONALSYMPTOM.parquet ===
["'PcrKey'", "'eSituation_10'", "'eSituation_10PN'"]

=== FACTPCRA

In [33]:
import duckdb

db_path = ROOT / "2024_duckdb" / "nemsis_2024.duckdb"

con = duckdb.connect(db_path)

In [34]:
import os
import re

parquet_dir = ROOT / "Main_Parquet_files"

files = [
    f for f in os.listdir(parquet_dir)
    if os.path.isfile(os.path.join(parquet_dir, f))
    and f.endswith(".parquet")
    and not f.startswith("._")
    and f != ".DS_Store"
]

# drop all views
views = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type='VIEW'
""").fetchall()

for (v,) in views:
    con.execute(f'DROP VIEW IF EXISTS "{v}"')


def qident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def clean_col_name(c: str) -> str:
    """
    Strips wrapping single or double quotes from column names.
    Example:
        "'eDispatch_01'"  ->  eDispatch_01
        '"eDispatch_01"'  ->  eDispatch_01
    """
    c = c.strip()

    if len(c) >= 2:
        if (c[0] == "'" and c[-1] == "'") or (c[0] == '"' and c[-1] == '"'):
            c = c[1:-1].strip()

    return c


for file in files:
    view_name = os.path.splitext(file)[0]
    full_path = os.path.join(parquet_dir, file)

    desc = con.execute(f"""
        DESCRIBE SELECT * FROM read_parquet('{full_path}')
    """).fetchall()

    raw_col_names = [row[0] for row in desc]

    # clean column names
    cleaned_col_names = [clean_col_name(c) for c in raw_col_names]

    # Optional: ensure no duplicates after cleaning
    seen = {}
    final_names = []
    for name in cleaned_col_names:
        if name not in seen:
            seen[name] = 1
            final_names.append(name)
        else:
            seen[name] += 1
            final_names.append(f"{name}_dup{seen[name]}")

    select_list = ",\n  ".join(
        [
            f'CAST(src.{qident(orig)} AS VARCHAR) AS {qident(new)}'
            for orig, new in zip(raw_col_names, final_names)
        ]
    )

    con.execute(f"""
        CREATE OR REPLACE VIEW {qident(view_name)} AS
        SELECT
          {select_list}
        FROM read_parquet('{full_path}') AS src;
    """)

    print("Created view:", view_name)

Created view: FactPcreOutcomeHospDiag
Created view: FACTPCRMEDICATION
Created view: FACTPCRARRESTRHYTHMDESTINATION
Created view: FACTPCRSCENEDELAY
Created view: PCRMEDCOMPGROUP
Created view: FACTPCRPRIMARYSYMPTOM
Created view: FACTPCRPROTOCOL
Created view: FACTPCRTRANSPORTDELAY
Created view: PCRPROCCOMPGROUP
Created view: FactPcreOutcomeEDProc
Created view: ESITUATION_11REF
Created view: FACTPCRWORKRELATEDEXPOSURE
Created view: FACTPCRTRAUMACRITERIA
Created view: FACTPCRALCOHOLDRUGUSEINDICATOR
Created view: FACTPCRARRESTROSC
Created view: EINJURY_01REF
Created view: Pub_PCRevents
Created view: PCRPATIENTRACEGROUP
Created view: PCRVITALGLASGOWQUALIFIERGROUP
Created view: EPROCEDURES_03REF
Created view: PCRVITALECGINTERPRETATIONGROUP
Created view: FACTPCRRESPONSEDELAY
Created view: FACTPCRBARRIERTOCARE
Created view: FACTPCRPRIMARYIMPRESSION
Created view: FACTPCRTURNAROUNDDELAY
Created view: FACTPCRARRESTCPRPROVIDED
Created view: FactPcreOutcomeEDDiag
Created view: FACTPCRDISPATCHDELAY
Cr

In [35]:
# Creating stroke_pub_pcr_events view (dispatch stroke/cva code) - 2024 names as-is
con.execute("""
CREATE OR REPLACE VIEW stroke_pub_pcr_events AS
SELECT *
FROM pub_pcrevents
WHERE TRIM(CAST(eDispatch_01 AS VARCHAR)) = '2301067'
""")

# Get all tables/views in main schema
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type IN ('BASE TABLE', 'VIEW')
""").fetchall()

# Tables/views having PcrKey
tables_with_pcrkey = []

for (table_name,) in tables:
    cols = [col[0] for col in con.execute(f"""
        SELECT name
        FROM pragma_table_info('{table_name}')
        ORDER BY cid
    """).fetchall()]

    # matching both common casings just in case
    if "PcrKey" in cols or "pcrKey" in cols:
        tables_with_pcrkey.append(table_name)

# Removing tables/views you don't want included
tables_to_exclude = {"stroke_pub_pcr_events", "pub_pcrevents"}
tables_with_pcrkey = [t for t in tables_with_pcrkey if t not in tables_to_exclude]

print("Count tables/views with PcrKey:", len(tables_with_pcrkey))
print("Example:", tables_with_pcrkey[:20])

Count tables/views with PcrKey: 37
Example: ['ComputedElements', 'FACTPCRADDITIONALRESPONSEMODE', 'FACTPCRADDITIONALSYMPTOM', 'FACTPCRADDITIONALTRANSPORTMODE', 'FACTPCRALCOHOLDRUGUSEINDICATOR', 'FACTPCRARRESTCPRPROVIDED', 'FACTPCRARRESTRESUSCITATION', 'FACTPCRARRESTRHYTHMDESTINATION', 'FACTPCRARRESTROSC', 'FACTPCRARRESTWITNESS', 'FACTPCRBARRIERTOCARE', 'FACTPCRCAUSEOFINJURY', 'FACTPCRDESTINATIONREASON', 'FACTPCRDESTINATIONTEAM', 'FACTPCRDISPATCHDELAY', 'FactPcreOutcomeEDDiag', 'FactPcreOutcomeEDProc', 'FactPcreOutcomeHospDiag', 'FactPcreOutcomeHospProc', 'FACTPCRHOSPITALCAPABILITY']


In [36]:
# Creating stroke version of tables having PcrKey

for table_name in tables_with_pcrkey:
    con.execute(f"""
        CREATE OR REPLACE VIEW stroke_{table_name} AS
        SELECT t.*
        FROM "{table_name}" t
        JOIN stroke_pub_pcr_events s
          ON t.PcrKey = s.PcrKey;
    """)

In [37]:
tables = con.execute("SHOW TABLES").fetchall()

for i,table in enumerate(tables):
    print(i)
    print(table)

0
('ComputedElements',)
1
('EINJURY_01REF',)
2
('EPROCEDURES_03REF',)
3
('ESITUATION_09REF',)
4
('ESITUATION_10REF',)
5
('ESITUATION_11REF',)
6
('ESITUATION_12REF',)
7
('FACTPCRADDITIONALRESPONSEMODE',)
8
('FACTPCRADDITIONALSYMPTOM',)
9
('FACTPCRADDITIONALTRANSPORTMODE',)
10
('FACTPCRALCOHOLDRUGUSEINDICATOR',)
11
('FACTPCRARRESTCPRPROVIDED',)
12
('FACTPCRARRESTRESUSCITATION',)
13
('FACTPCRARRESTRHYTHMDESTINATION',)
14
('FACTPCRARRESTROSC',)
15
('FACTPCRARRESTWITNESS',)
16
('FACTPCRBARRIERTOCARE',)
17
('FACTPCRCAUSEOFINJURY',)
18
('FACTPCRDESTINATIONREASON',)
19
('FACTPCRDESTINATIONTEAM',)
20
('FACTPCRDISPATCHDELAY',)
21
('FACTPCRHOSPITALCAPABILITY',)
22
('FACTPCRINJURYRISKFACTOR',)
23
('FACTPCRMEDICATION',)
24
('FACTPCRPRIMARYIMPRESSION',)
25
('FACTPCRPRIMARYSYMPTOM',)
26
('FACTPCRPROCEDURE',)
27
('FACTPCRPROTOCOL',)
28
('FACTPCRRESPONSEDELAY',)
29
('FACTPCRSCENEDELAY',)
30
('FACTPCRSECONDARYIMPRESSION',)
31
('FACTPCRTIME',)
32
('FACTPCRTRANSPORTDELAY',)
33
('FACTPCRTRAUMACRITERIA',)
3

In [38]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type IN ('BASE TABLE', 'VIEW')
""").fetchall()

tables_with_pcrkey = {}

for (table_name,) in tables:
    cols = [col[0] for col in con.execute(f"""
        SELECT name
        FROM pragma_table_info('{table_name}')
        ORDER BY cid
    """).fetchall()]

    if "PcrKey" in cols or "pcrKey" in cols:
        tables_with_pcrkey[table_name] = cols  # storing full column list

# print nicely
for k, v in tables_with_pcrkey.items():
    print(f"\n{k}")
    print(v)

print("\nTotal tables with PcrKey:", len(tables_with_pcrkey))


ComputedElements
['PcrKey', 'USCensusRegion', 'USCensusDivision', 'NasemsoRegion', 'Urbanicity', 'ageinyear', 'EMSDispatchCenterTimeSec', 'EMSChuteTimeMin', 'EMSSystemResponseTimeMin', 'EMSSceneResponseTimeMin', 'EMSSceneTimeMin', 'EMSSceneToPatientTimeMin', 'EMSTransportTimeMin', 'EMSTotalCallTimeMin']

FACTPCRADDITIONALRESPONSEMODE
['PcrKey', 'eResponse_24']

FACTPCRADDITIONALSYMPTOM
['PcrKey', 'eSituation_10', 'eSituation_10PN']

FACTPCRADDITIONALTRANSPORTMODE
['PcrKey', 'eDisposition_18']

FACTPCRALCOHOLDRUGUSEINDICATOR
['PcrKey', 'eHistory_17']

FACTPCRARRESTCPRPROVIDED
['PcrKey', 'eArrest_09']

FACTPCRARRESTRESUSCITATION
['PcrKey', 'eArrest_03']

FACTPCRARRESTRHYTHMDESTINATION
['PcrKey', 'eArrest_17']

FACTPCRARRESTROSC
['PcrKey', 'eArrest_12']

FACTPCRARRESTWITNESS
['PcrKey', 'eArrest_04']

FACTPCRBARRIERTOCARE
['PcrKey', 'eHistory_01']

FACTPCRCAUSEOFINJURY
['PcrKey', 'eInjury_01']

FACTPCRDESTINATIONREASON
['PcrKey', 'eDisposition_20']

FACTPCRDESTINATIONTEAM
['eDisposition_2

In [39]:
strk_tbl_cat_feat_2024 = {
    "stroke_FACTPCRADDITIONALRESPONSEMODE": ["eResponse_24"],
    "stroke_FACTPCRADDITIONALSYMPTOM": ["eSituation_10"],
    "stroke_FACTPCRADDITIONALTRANSPORTMODE": ["eDisposition_18"],
    "stroke_FACTPCRALCOHOLDRUGUSEINDICATOR": ["eHistory_17"],
    "stroke_FACTPCRARRESTCPRPROVIDED": ["eArrest_09"],
    "stroke_FACTPCRARRESTRESUSCITATION": ["eArrest_03"],
    "stroke_FACTPCRARRESTRHYTHMDESTINATION": ["eArrest_17"],
    "stroke_FACTPCRARRESTROSC": ["eArrest_12"],
    "stroke_FACTPCRARRESTWITNESS": ["eArrest_04"],
    "stroke_FACTPCRBARRIERTOCARE": ["eHistory_01"],
    "stroke_FACTPCRCAUSEOFINJURY": ["eInjury_01"],
    "stroke_FACTPCRDESTINATIONREASON": ["eDisposition_20"],
    "stroke_FACTPCRDESTINATIONTEAM": ["eDisposition_24"],
    "stroke_FACTPCRDISPATCHDELAY": ["eResponse_08"],
    "stroke_FACTPCRHOSPITALCAPABILITY": ["eDisposition_23"],
    "stroke_FACTPCRINJURYRISKFACTOR": ["eInjury_04"],

    # MEDICATION (many columns)
    "stroke_FACTPCRMEDICATION": [
        "PcrMedicationKey",
        "eMedications_02",
        "eMedications_03",
        "eMedications_04",
        "eMedications_06",
        "eMedications_07",
        "eMedications_10",
    ],

    "stroke_FACTPCRPRIMARYIMPRESSION": ["eSituation_11"],
    "stroke_FACTPCRPRIMARYSYMPTOM": ["eSituation_09"],

    # PROCEDURE (many columns)
    "stroke_FACTPCRPROCEDURE": [
        "PcrProcedureKey",
        "eProcedures_02",
        "eProcedures_03",
        "eProcedures_06",
        "eProcedures_08",
        "eProcedures_10",
    ],

    # PROTOCOL (many columns)
    "stroke_FACTPCRPROTOCOL": [
        "eProtocols_01",
        "eProtocols_02",
    ],

    "stroke_FACTPCRRESPONSEDELAY": ["eResponse_09"],
    "stroke_FACTPCRSCENEDELAY": ["eResponse_10"],
    "stroke_FACTPCRSECONDARYIMPRESSION": ["eSituation_12"],
    "stroke_FACTPCRTRANSPORTDELAY": ["eResponse_11"],
    "stroke_FACTPCRTRAUMACRITERIA": ["eInjury_03"],
    "stroke_FACTPCRTURNAROUNDDELAY": ["eResponse_12"],
    "stroke_FACTPCRWORKRELATEDEXPOSURE": ["eOther_05"],

    # Outcomes
    "stroke_FactPcreOutcomeEDDiag": ["eOutcome_10"],
    "stroke_FactPcreOutcomeEDProc": ["eOutcome_09"],
    "stroke_FactPcreOutcomeHospDiag": ["eOutcome_13"],
    "stroke_FactPcreOutcomeHospProc": ["eOutcome_12"],

    "stroke_pub_pcr_events" : ["eScene_09"],
    "stroke_ComputedElements":["USCensusRegion", "USCensusDivision", "NasemsoRegion", "Urbanicity"],
    "stroke_PCRPATIENTRACEGROUP": ["ePatient_14"],

    # Vitals
    "stroke_FACTPCRVITAL" : ["PcrVitalKey","eVitals_02", "eVitals_04", "eVitals_08", "eVitals_19", "eVitals_20", "eVitals_21", "eVitals_26", "eVitals_29", "eVitals_30", "eVitals_31"]
}


strk_tbl_num_feat_2024 = {
    "stroke_FACTPCRVITAL" : ["eVitals_06", "eVitals_10", "eVitals_12", "eVitals_14", "eVitals_18", "eVitals_27"],
    "stroke_FACTPCRPROCEDURE" : ["eMedications_05"]
}

In [40]:
# Creating strk_pubpcr_computed_elmts table

sql = '''
CREATE OR REPLACE VIEW strk_pubpcr_computed_elmts AS
SELECT *
FROM stroke_pub_pcr_events sppe
LEFT JOIN stroke_ComputedElements sce ON sppe.PcrKey = sce.PcrKey
'''

con.execute(sql)

In [41]:
# Aggregating numeric vitals on PcrKey and attaching to stroke_master
# - Skipping NEMSIS null/special codes: 7701001, 7701003, 7701005, 8801005, 8801019, 8801023
# - Handling eVitals_18 (blood glucose) separately because it can contain 'High'/'Low'

BAD_CODES = ("7701001", "7701003", "7701005", "8801005", "8801019", "8801023")

numeric_vitals = [
    "eVitals_06",  # SBP
    "eVitals_10",  # Heart Rate
    "eVitals_12",  # Pulse Ox
    "eVitals_14",  # Respiratory Rate
    "eVitals_27",  # Pain Scale
    "eVitals_18",  # Blood Glucose (special)
]

# 1) Aggregate numeric vitals EXCEPT eVitals_18
for val in [v for v in numeric_vitals if v != "eVitals_18"]:
    con.execute(f"""
        CREATE OR REPLACE VIEW agg_{val} AS
        SELECT
            TRIM(CAST(PcrKey AS VARCHAR)) AS PcrKey,
            AVG(TRY_CAST(TRIM({val}) AS DOUBLE)) AS avg_{val}
        FROM stroke_FACTPCRVITAL
        WHERE
            {val} IS NOT NULL
            AND TRIM(CAST({val} AS VARCHAR)) <> ''
            AND TRIM(CAST({val} AS VARCHAR)) NOT IN {BAD_CODES}
            AND TRY_CAST(TRIM({val}) AS DOUBLE) IS NOT NULL
        GROUP BY 1
    """)

# 2) Aggregate eVitals_18 (glucose) separately
# Accepting numeric strings only; excluding 'High'/'Low' and BAD_CODES
con.execute(f"""
    CREATE OR REPLACE VIEW agg_eVitals_18 AS
    SELECT
        TRIM(CAST(PcrKey AS VARCHAR)) AS PcrKey,
        AVG(TRY_CAST(TRIM(eVitals_18) AS DOUBLE)) AS avg_eVitals_18,
        MIN(TRY_CAST(TRIM(eVitals_18) AS DOUBLE)) AS min_eVitals_18,
        MAX(TRY_CAST(TRIM(eVitals_18) AS DOUBLE)) AS max_eVitals_18
    FROM stroke_FACTPCRVITAL
    WHERE
        eVitals_18 IS NOT NULL
        AND TRIM(CAST(eVitals_18 AS VARCHAR)) <> ''
        AND TRIM(CAST(eVitals_18 AS VARCHAR)) NOT IN {BAD_CODES}
        AND LOWER(TRIM(CAST(eVitals_18 AS VARCHAR))) NOT IN ('high', 'low')
        AND TRY_CAST(TRIM(eVitals_18) AS DOUBLE) IS NOT NULL
    GROUP BY 1
""")

# 3) Attach aggregated vitals to stroke_master (progressively, like your other attach steps)
current = "strk_pubpcr_computed_elmts"

for val in [v for v in numeric_vitals if v != "eVitals_18"]:
    next_view = f'{current}__avg_{val}'
    con.execute(f"""
        CREATE OR REPLACE VIEW "{next_view}" AS
        SELECT m.*, a.avg_{val}
        FROM "{current}" m
        LEFT JOIN agg_{val} a
          ON TRIM(CAST(m.PcrKey AS VARCHAR)) = a.PcrKey
    """)
    current = next_view

# attach glucose aggregates
next_view = f'{current}__avg_eVitals_18'
con.execute(f"""
    CREATE OR REPLACE VIEW "{next_view}" AS
    SELECT m.*, g.avg_eVitals_18, g.min_eVitals_18, g.max_eVitals_18
    FROM "{current}" m
    LEFT JOIN agg_eVitals_18 g
      ON TRIM(CAST(m.PcrKey AS VARCHAR)) = g.PcrKey
""")
current = next_view

print("Master with aggregated numeric vitals:", current)

con.execute(f"ALTER VIEW {current} RENAME TO stroke_master;")


Master with aggregated numeric vitals: strk_pubpcr_computed_elmts__avg_eVitals_06__avg_eVitals_10__avg_eVitals_12__avg_eVitals_14__avg_eVitals_27__avg_eVitals_18


In [42]:
NV = {"7701001", "7701003", "7701005", ".", "not recorded", "not applicable", "unknown"}
NV_NORM = {v.strip().lower() for v in NV}

def get_combo_keys(con, table, cat_col, *, pcr_col="PcrKey", delim="__", nv_codes=None):
    nv_norm = NV_NORM if nv_codes else set()

    nv_sql = ",".join(f"'{v}'" for v in sorted(nv_norm))

    rows = con.execute(f"""
        SELECT
            TRIM(CAST({pcr_col} AS VARCHAR)) AS pcr_key,
            LIST(
                DISTINCT TRIM(CAST({cat_col} AS VARCHAR))
                ORDER BY TRIM(CAST({cat_col} AS VARCHAR))
            ) AS cats
        FROM {table}
        WHERE
            -- skip invalid PCR keys
            {pcr_col} IS NOT NULL
            AND TRIM(CAST({pcr_col} AS VARCHAR)) <> ''
            AND LOWER(TRIM(CAST({pcr_col} AS VARCHAR))) NOT IN ({nv_sql})

            -- skip invalid category values
            AND {cat_col} IS NOT NULL
            AND TRIM(CAST({cat_col} AS VARCHAR)) <> ''
            AND LOWER(TRIM(CAST({cat_col} AS VARCHAR))) NOT IN ({nv_sql})

        GROUP BY 1
    """).fetchall()

    combos = set()
    for _, cats in rows:
        if not cats:
            continue
        clean = [
            c for c in cats
            if c and c.strip() and c.strip().lower() not in nv_norm
        ]
        if clean:
            combos.add(delim.join(clean))

    return sorted(combos)

In [43]:
def create_encoding_view(con, view_name, combo_keys, offset):
    values = []
    for i, k in enumerate(combo_keys):
        k_escaped = k.replace("'", "''")
        enc_id = offset + i + 1
        values.append(f"({enc_id}, '{k_escaped}')")

    rows = ",\n".join(values)

    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT *
        FROM (VALUES
            {rows}
        ) AS t(enc_id, combo_key);
    """)

In [44]:
def create_pcr_id_view(
    con,
    *,
    out_view,
    src_table,
    cat_col,
    enc_view,
    out_id_col,
    pcr_col="PcrKey",
    delim="__",
    not_codes=(),
):
    con.execute(f"""
        CREATE OR REPLACE VIEW {out_view} AS
        SELECT
            p.{pcr_col},
            e.enc_id AS {out_id_col}
        FROM (
            SELECT
                {pcr_col},
                string_agg(
                    DISTINCT TRIM(CAST({cat_col} AS VARCHAR)),
                    '{delim}'
                    ORDER BY TRIM(CAST({cat_col} AS VARCHAR))
                ) AS combo_key
            FROM {src_table}
            WHERE {cat_col} IS NOT NULL
              AND TRIM(CAST({cat_col} AS VARCHAR)) NOT IN {not_codes}
            GROUP BY {pcr_col}
        ) p
        JOIN {enc_view} e
          ON p.combo_key = e.combo_key;
    """)

In [45]:
def attach_to_master(con, *, base_master, out_master, pcr_id_view, out_id_col, pcr_col="PcrKey"):
    con.execute(f"""
        CREATE OR REPLACE VIEW {out_master} AS
        SELECT
            m.*,
            f.{out_id_col}
        FROM {base_master} m
        LEFT JOIN {pcr_id_view} f
          ON m.{pcr_col} = f.PcrKey;
    """)

In [46]:
offset = 100

for key,value in strk_tbl_cat_feat_2024.items():
    for val in value:
        print(key,val)
        combo_keys = get_combo_keys(
        con,
        table=key,
        cat_col=val,
        nv_codes=NV
        )
        create_encoding_view(con, f"lookup_{val}", combo_keys,offset)
        offset += 100
 



stroke_FACTPCRADDITIONALRESPONSEMODE eResponse_24


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

stroke_FACTPCRADDITIONALSYMPTOM eSituation_10
stroke_FACTPCRADDITIONALTRANSPORTMODE eDisposition_18


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

stroke_FACTPCRALCOHOLDRUGUSEINDICATOR eHistory_17


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

stroke_FACTPCRVITAL eVitals_30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

stroke_FACTPCRVITAL eVitals_31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [47]:
NOT_CODES = NV = ("7701001", "7701003", "7701005", ".", "not recorded", "not applicable", "unknown")


for key, value in strk_tbl_cat_feat_2024.items():
    for val in value:
        create_pcr_id_view(
            con,
            out_view=f"coded_{val}",
            src_table=key,
            cat_col=val,
            enc_view=f"lookup_{val}",
            out_id_col=f"{val}_ID",
            pcr_col="PcrKey",
            delim="__",
            not_codes=NOT_CODES,
        )

In [48]:
current = "stroke_master"   

for table, cols in strk_tbl_cat_feat_2024.items():
    for col in cols:
        next_view = f"{current}__{col}"   

        attach_to_master(
            con,
            base_master=current,
            out_master=next_view,
            pcr_id_view=f"coded_{col}",
            out_id_col=f"{col}_ID",
        )

        current = next_view

final_master = current
print("Final master view:", final_master)

Final master view: stroke_master__eResponse_24__eSituation_10__eDisposition_18__eHistory_17__eArrest_09__eArrest_03__eArrest_17__eArrest_12__eArrest_04__eHistory_01__eInjury_01__eDisposition_20__eDisposition_24__eResponse_08__eDisposition_23__eInjury_04__PcrMedicationKey__eMedications_02__eMedications_03__eMedications_04__eMedications_06__eMedications_07__eMedications_10__eSituation_11__eSituation_09__PcrProcedureKey__eProcedures_02__eProcedures_03__eProcedures_06__eProcedures_08__eProcedures_10__eProtocols_01__eProtocols_02__eResponse_09__eResponse_10__eSituation_12__eResponse_11__eInjury_03__eResponse_12__eOther_05__eOutcome_10__eOutcome_09__eOutcome_13__eOutcome_12__eScene_09__USCensusRegion__USCensusDivision__NasemsoRegion__Urbanicity__ePatient_14__PcrVitalKey__eVitals_02__eVitals_04__eVitals_08__eVitals_19__eVitals_20__eVitals_21__eVitals_26__eVitals_29__eVitals_30__eVitals_31


In [49]:
# Renaming the final master view

con.execute(f"ALTER VIEW {final_master} RENAME TO stroke_master_final;")


In [50]:
con.execute("DESCRIBE stroke_master_final").fetchall()

[('PcrKey', 'VARCHAR', 'YES', None, None, None),
 ('eDispatch_01', 'VARCHAR', 'YES', None, None, None),
 ('eDispatch_02', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_14', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_01', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_02', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_05', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_07', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_11', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_16', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_18', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_20', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_21', 'VARCHAR', 'YES', None, None, None),
 ('eArrest_22', 'VARCHAR', 'YES', None, None, None),
 ('eDisposition_19', 'VARCHAR', 'YES', None, None, None),
 ('eDisposition_16', 'VARCHAR', 'YES', None, None, None),
 ('eDisposition_21', 'VARCHAR', 'YES', None, None, None),
 ('eDisposition_22', 'VARCHAR', 'YES', None, None, None),
 ('eDisposition_32', 'VARC

In [51]:
con.execute(f"""
    COPY (
        SELECT
            TRIM(CAST(PcrKey AS VARCHAR)) AS PcrKey,
            * EXCLUDE (PcrKey)
        FROM stroke_master_final
        WHERE TRIM(CAST(PcrKey AS VARCHAR)) NOT IN ('', '.')
    )
    TO '{ROOT / "Data_Files" / "2024_stroke_master_final.parquet"}'
    (FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [52]:
import os

out_dir = ROOT / "Data_Files" / "2024_lookups_parquet"
os.makedirs(out_dir, exist_ok=True)

enc_views = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'VIEW'
      AND table_name LIKE 'lookup_%'
""").fetchall()

for (v,) in enc_views:
    out_path = os.path.join(out_dir, f"{v}.parquet")
    con.execute(f"""
        COPY (SELECT * FROM "{v}")
        TO '{out_path}'
        (FORMAT PARQUET);
    """)
    print("Wrote:", out_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eArrest_03.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eArrest_04.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eArrest_09.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eArrest_12.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eArrest_17.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eDisposition_18.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eDisposition_20.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_lookups_parquet/lookup_eDisposition_23.parquet
Wrote: /Users/heda/Deskto

In [53]:

out_dir = ROOT / "Data_Files" / "2024_coded_parquet"
os.makedirs(out_dir, exist_ok=True)

coded_views = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'VIEW'
      AND table_name LIKE 'coded_%'
""").fetchall()

for (v,) in coded_views:
    out_path = os.path.join(out_dir, f"{v}.parquet")
    con.execute(f"""
        COPY (SELECT * FROM "{v}")
        TO '{out_path}'
        (FORMAT PARQUET);
    """)
    print("Wrote:", out_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eArrest_03.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eArrest_04.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eArrest_09.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eArrest_12.parquet
Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eArrest_17.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_eDisposition_18.parquet


...

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_Urbanicity.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_USCensusDivision.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote: /Users/heda/Desktop/GitHub/NEMSIS_2024_Counterfactuals/Data_Files/2024_coded_parquet/coded_USCensusRegion.parquet


In [54]:
import pandas as pd
import numpy as np

df_original = pd.read_parquet(
    ROOT / "Data_Files" / "2024_stroke_master_final.parquet"
)

# textual NA values (case-insensitive)
TEXT_NA = {
    "not recorded",
    "not applicable",
    "unknown"
}

# NEMSIS NOT codes (as strings)
NOT_CODES = {"7701001", "7701003", "7701005"}

# extra placeholders that should be treated as missing after trimming
EXTRA_NA = {"."}

def clean_object_column(s: pd.Series) -> pd.Series:
    # converting to string and stripping whitespace
    s_clean = s.astype("string").str.strip()

    # lowercasing for case-insensitive comparisons
    s_lower = s_clean.str.lower()

    # building NA mask
    na_mask = (
        s_lower.isin(TEXT_NA) |
        s_clean.isin(NOT_CODES) |
        s_clean.isin(EXTRA_NA) |
        s_clean.isna() |
        (s_clean == "")
    )

    return s_clean.mask(na_mask)

obj_cols = df_original.select_dtypes(include=["object", "string"]).columns

df_cleaned = df_original.copy()
df_cleaned[obj_cols] = df_cleaned[obj_cols].apply(clean_object_column)

# checks
for code in NOT_CODES:
    assert not (df_cleaned[obj_cols] == code).any().any()
assert not (df_cleaned[obj_cols] == ".").any().any()

# write cleaned parquet
df_cleaned.to_parquet(
    ROOT / "Data_Files" / "stroke_master_final_cleaned.parquet",
    index=False
)